# LC 322 — Coin Change
**Day 54 | Mixed Review Sprint | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Build a 1-D DP table where
<code>dp[i]</code> = fewest coins to make amount <code>i</code>.
Seed <code>dp[0]=0</code>, everything else infinity.
For every amount from 1 to target, try every coin:
<code>dp[i] = min(dp[i], dp[i-coin]+1)</code>.
</div>

## Official Problem Statement

You are given an integer array `coins` representing coins
of different denominations and an integer `amount`
representing a total amount of money.

Return the fewest number of coins that you need to make
up that amount. If that amount of money cannot be made up
by any combination of the coins, return `-1`.

You may assume that you have an **infinite number** of
each kind of coin.

**Constraints:**
- `1 <= coins.length <= 12`
- `1 <= coins[i] <= 2^31 - 1`
- `0 <= amount <= 10^4`

## What This Is Actually Asking

Given unlimited coins of given denominations, find the
minimum number of coins to reach exactly `amount`.
Greedy (always pick largest coin) fails for some coin
sets, so we need dynamic programming.
DP builds up optimal answers for smaller amounts first,
then uses them to solve larger amounts — classic
bottom-up approach.
If no combination reaches `amount`, return -1.

## Walk Through an Example by Hand

```
coins = [1, 5, 6, 9]   amount = 11

dp = [0, inf, inf, inf, inf, inf, inf, inf, inf, inf,
      inf, inf]    (indices 0..11)

i=1:  coin=1 -> dp[1-1]+1=1  dp[1]=1
i=2:  coin=1 -> dp[1]+1=2    dp[2]=2
i=3:  coin=1 -> 3            dp[3]=3
i=4:  coin=1 -> 4            dp[4]=4
i=5:  coin=1 -> 5
      coin=5 -> dp[0]+1=1    dp[5]=1
i=6:  coin=1 -> dp[5]+1=2
      coin=5 -> dp[1]+1=2
      coin=6 -> dp[0]+1=1    dp[6]=1
...
i=11: coin=1 -> dp[10]+1
      coin=5 -> dp[6]+1=2
      coin=6 -> dp[5]+1=2
      coin=9 -> dp[2]+1=3
      best = 2               dp[11]=2

Answer: 2  (5+6 or 6+5)
```

## The Picture

```
coins=[1,2,5]   amount=6

Index: 0   1   2   3   4   5   6
Init:  0  inf inf inf inf inf inf

After processing each amount:

  i=1: try coin=1 -> dp[0]+1=1
       dp: [0, 1, inf, inf, inf, inf, inf]

  i=2: try coin=1 -> dp[1]+1=2
       try coin=2 -> dp[0]+1=1  <- better!
       dp: [0, 1,  1,  inf, inf, inf, inf]

  i=3: try coin=1 -> dp[2]+1=2
       try coin=2 -> dp[1]+1=2
       dp: [0, 1,  1,  2,   inf, inf, inf]

  i=4: try coin=1 -> 3
       try coin=2 -> dp[2]+1=2  <- better!
       dp: [0, 1,  1,  2,   2,   inf, inf]

  i=5: try coin=1 -> 3
       try coin=2 -> dp[3]+1=3
       try coin=5 -> dp[0]+1=1  <- best!
       dp: [0, 1,  1,  2,   2,   1,   inf]

  i=6: try coin=1 -> dp[5]+1=2
       try coin=2 -> dp[4]+1=3
       try coin=5 -> dp[1]+1=2
       dp: [0, 1,  1,  2,   2,   1,   2  ]

  Answer = dp[6] = 2   (coins: 1+5 or 5+1)
```

## When To Use This Pattern

- When asked for **minimum / maximum count** to reach a
  target with unlimited reuse, think **unbounded DP**.
- When greedy fails because optimal substructure depends
  on coin denominations, think **bottom-up DP table**.
- When subproblems overlap (amount 11 needs amount 6
  which was already computed), think **memoisation or
  tabulation**.
- When the state space is `amount` and choices are
  `coins`, think **1-D DP** iterating over amounts then
  coins.
- When answer could be impossible, think **initialise
  with infinity and check at the end**.

## The Approach

Create a DP array of size `amount+1` filled with
infinity, then set `dp[0] = 0` because zero coins
make zero amount. Iterate every amount from 1 to
`amount`; for each, try every coin: if `coin <= i`,
update `dp[i] = min(dp[i], dp[i-coin] + 1)`. After
the loops, return `dp[amount]` if it's not infinity,
otherwise return -1.

In [ ]:
from typing import List
from collections import defaultdict, deque

In [ ]:
def test_harness(func):
    cases = [
        # (coins, amount, expected)
        ([1, 5, 6, 9], 11, 2),
        ([1, 2, 5],    11, 3),
        ([2],           3, -1),
        ([1],           0,  0),
        ([1],           1,  1),
        ([186, 419, 83, 408], 6249, 20),
    ]
    passed = 0
    for coins, amount, expected in cases:
        result = func(coins, amount)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(
                f"{status} | coins={coins} "
                f"amount={amount} | "
                f"got={result} expected={expected}"
            )
    print(f"\nResults: {passed}/{len(cases)} passed")
    if passed == len(cases):
        print("All tests PASSED!")

In [ ]:
def coin_change(coins: List[int], amount: int) -> int:
    """
    Return fewest coins to make amount, or -1 if impossible.

    Strategy: bottom-up 1-D DP.
      dp[0] = 0
      dp[i] = min(dp[i-coin] + 1) for all valid coins
      Final: dp[amount] if != inf else -1

    Args:
        coins:  list of coin denominations
        amount: target amount

    Returns:
        int: minimum coins needed, or -1
    """
    print(f"[DEBUG] coins={coins} amount={amount}")

    INF = float('inf')
    dp = [INF] * (amount + 1)
    dp[0] = 0

    print(f"[DEBUG] initial dp[:6]={dp[:6]}")

    for i in range(1, amount + 1):
        for coin in coins:
            if coin <= i and dp[i - coin] + 1 < dp[i]:
                dp[i] = dp[i - coin] + 1

    print(f"[DEBUG] dp[amount]={dp[amount]}")
    return dp[amount] if dp[amount] != INF else -1


pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(coin_change)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute Force (recursion) | O(S^n) | O(n) | Exponential without memo |
| Memoised DFS | O(S*n) | O(S) | S=amount, n=coins |
| Optimal (bottom-up DP) | O(S*n) | O(S) | Constant factor better |

## Real World Connection

At **Citi**, payment routing engines choose the minimum
number of settlement hops across correspondent banks to
reach a target currency — structurally identical to
Coin Change. **AWS Lambda** billing uses denomination-
like rounding to the nearest 1ms, 100ms, or 1s tier;
optimising compute units resembles this DP. In **data
engineering**, partition-size optimisation picks the
fewest chunk sizes (coins) to cover a dataset target
without waste. This pattern also appears in any
resource allocation problem that allows reuse.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra